# 💻 Unidad 4: Material Complementario - Práctica
## Módulo 04 - Arquitecturas ML End-to-End
### Laboratorio (Herramientas) - Universidad del Aconcagua

---

## 🎯 Objetivos de la Práctica

En esta práctica vas a:

1. ✅ Implementar un pipeline ML completo (Bronze → Silver → Gold)
2. ✅ Aplicar feature engineering
3. ✅ Entrenar y evaluar modelo con MLflow
4. ✅ Validar modelo antes de deployment
5. ✅ Configurar monitoreo en producción
6. ✅ Documentar pipeline end-to-end

---

### 📋 Proyecto

**Caso de Uso**: Predicción de Churn de Clientes

**Pipeline Completo**:
* 🟫 Bronze: Ingestión de datos crudos
* 🥈 Silver: Limpieza y validación
* 🥇 Gold: Feature engineering
* 🏋️‍♂️ Training: Entrenar con MLflow
* ✅ Validation: Tests de calidad
* 📦 Registry: Registrar modelo
* 📊 Monitoring: Configurar alertas

---

### ⏱️ Duración Estimada: 90 minutos

## 🛠️ Setup: Instalación de Librerías

Instalamos MLflow y scikit-learn:

In [0]:
# Instalar librerías
%pip install mlflow scikit-learn

print("✅ Librerías instaladas")

In [0]:
# Imports
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías importadas")

---

## 🟫 Paso 1: Data Ingestion (Bronze Layer)

**Objetivo**: Ingestar datos crudos sin transformaciones

**Características**:
* Datos tal cual llegan de la fuente
* Sin validaciones ni limpieza
* 5,000 clientes simulados
* Features: age, tenure_months, monthly_spend, num_products
* Target: churn (0/1)

In [0]:
print("🟫 Paso 1: Data Ingestion (Bronze)\n" + "="*60)

np.random.seed(42)
n_samples = 5000

data_bronze = pd.DataFrame({
    'customer_id': range(n_samples),
    'age': np.random.randint(18, 70, n_samples),
    'tenure_months': np.random.randint(1, 60, n_samples),
    'monthly_spend': np.random.uniform(20, 200, n_samples),
    'num_products': np.random.randint(1, 5, n_samples)
})

# Simular churn (target)
data_bronze['churn'] = ((data_bronze['tenure_months'] < 12) & 
                        (data_bronze['monthly_spend'] < 50)).astype(int)

print(f"✅ Datos crudos ingestados: {len(data_bronze):,} registros")
print(f"\nColumnas: {list(data_bronze.columns)}")
print(f"\nTasa de churn: {data_bronze['churn'].mean():.2%}")
print(f"\n💡 Bronze = datos SIN procesar (tal cual de la fuente)")

display(data_bronze.head())

---

## 🥈 Paso 2: Data Processing (Silver Layer)

**Objetivo**: Limpiar y validar datos

**Transformaciones**:
* Filtrar edades válidas (18-100)
* Eliminar gastos negativos o cero
* Validar integridad de datos

**Output**: Dataset limpio y validado

In [0]:
print("🥈 Paso 2: Data Processing (Silver)\n" + "="*60)

# Limpieza y validación
data_silver = data_bronze[
    (data_bronze['age'] >= 18) & 
    (data_bronze['age'] <= 100) &
    (data_bronze['monthly_spend'] > 0)
].copy()

registros_eliminados = len(data_bronze) - len(data_silver)

print(f"✅ Datos limpios: {len(data_silver):,} registros")
print(f"  Registros eliminados: {registros_eliminados}")
print(f"  % Conservado: {(len(data_silver)/len(data_bronze)):.1%}")

print(f"\n💡 Silver = datos LIMPIOS (validados y filtrados)")

display(data_silver.head())

---

## 🥇 Paso 3: Feature Engineering (Gold Layer)

**Objetivo**: Crear features derivadas para ML

**Features Nuevas**:
* `spend_per_product`: Gasto promedio por producto
* `is_young`: Cliente menor de 30 años
* `is_new_customer`: Tenure < 6 meses

**Output**: Dataset listo para entrenamiento

In [0]:
print("🥇 Paso 3: Feature Engineering (Gold)\n" + "="*60)

# Crear features derivadas
data_gold = data_silver.copy()

data_gold['spend_per_product'] = data_gold['monthly_spend'] / data_gold['num_products']
data_gold['is_young'] = (data_gold['age'] < 30).astype(int)
data_gold['is_new_customer'] = (data_gold['tenure_months'] < 6).astype(int)

features = ['age', 'tenure_months', 'monthly_spend', 'num_products', 
            'spend_per_product', 'is_young', 'is_new_customer']

print(f"✅ Features creadas: {len(features)}")
print(f"\nLista de features:")
for i, feat in enumerate(features, 1):
    print(f"  {i}. {feat}")

print(f"\n💡 Gold = datos ENRIQUECIDOS (con features de negocio)")

display(data_gold[features + ['churn']].head())

---

## 🏋️‍♂️ Paso 4: Training con MLflow

**Objetivo**: Entrenar modelo con experiment tracking

**Configuración**:
* Algoritmo: Random Forest
* Split: 80% train, 20% test
* Tracking: MLflow
* Métricas: Accuracy, F1, Precision, Recall

**MLflow Benefits**: Reproducibilidad y trazabilidad

In [0]:
print("🏋️‍♂️ Paso 4: Training con MLflow\n" + "="*60)

# Preparar datos
X = data_gold[features]
y = data_gold['churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n📊 Split de datos:")
print(f"  Train: {len(X_train):,} registros")
print(f"  Test: {len(X_test):,} registros")

# MLflow experiment tracking
mlflow.set_experiment("/Users/cortega@uda.edu.ar/churn_prediction_e2e")

with mlflow.start_run(run_name="rf_baseline") as run:
    # Entrenar
    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=42
    )
    model.fit(X_train, y_train)
    
    # Evaluar
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    
    # Log params
    mlflow.log_params({
        "n_estimators": 100,
        "max_depth": 10,
        "n_features": len(features)
    })
    
    # Log metrics
    mlflow.log_metrics({
        "accuracy": accuracy,
        "f1_score": f1,
        "precision": precision,
        "recall": recall
    })
    
    # Log model
    mlflow.sklearn.log_model(model, "model")
    
    run_id = run.info.run_id
    
    print(f"\n✅ Modelo entrenado y loggeado en MLflow")
    print(f"  Run ID: {run_id}")
    print(f"\n📊 Métricas:")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  F1-Score: {f1:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")

---

## ✅ Paso 5: Validation

**Objetivo**: Validar que el modelo cumple criterios de calidad

**Criterios de Validación**:
* Accuracy mínimo: 75%
* F1-Score mínimo: 0.70
* Sin errores en predicción

**Decisión**: Si pasa → Registry, si no → Reentrenar

In [0]:
print("✅ Paso 5: Validation\n" + "="*60)

# Criterios de validación
min_accuracy = 0.75
min_f1 = 0.70

print("\n📊 Criterios de validación:")
print(f"  Accuracy mínimo: {min_accuracy:.0%}")
print(f"  F1-Score mínimo: {min_f1:.2f}")

print("\n📊 Resultados del modelo:")
print(f"  Accuracy: {accuracy:.4f} ({accuracy:.1%})")
print(f"  F1-Score: {f1:.4f}")

# Validar
validation_passed = (accuracy >= min_accuracy) and (f1 >= min_f1)

if validation_passed:
    print("\n✅ VALIDACIÓN EXITOSA")
    print("  El modelo cumple todos los criterios")
    print("  🚀 Listo para Registry")
    deploy_approved = True
else:
    print("\n❌ VALIDACIÓN FALLIDA")
    if accuracy < min_accuracy:
        print(f"  ⚠️ Accuracy insuficiente: {accuracy:.1%} < {min_accuracy:.0%}")
    if f1 < min_f1:
        print(f"  ⚠️ F1-Score insuficiente: {f1:.4f} < {min_f1:.2f}")
    print("  🔄 Reentrenar con más datos o ajustar hiperparámetros")
    deploy_approved = False

---

## 📦 Paso 6: Model Registry

**Objetivo**: Registrar modelo en MLflow Registry

**Proceso**:
1. Registrar modelo desde el run
2. Asignar nombre descriptivo
3. Agregar metadata (versión, métricas)
4. Preparar para deployment

**Output**: Modelo versionado y listo para Staging

In [0]:
print("📦 Paso 6: Model Registry\n" + "="*60)

if deploy_approved:
    print("\n📊 Preparando registro del modelo...")
    
    # En producción real:
    # model_uri = f"runs:/{run_id}/model"
    # registered_model = mlflow.register_model(model_uri, "churn_predictor")
    
    model_info = {
        "nombre": "churn_predictor",
        "version": "1.0.0",
        "run_id": run_id,
        "accuracy": accuracy,
        "f1_score": f1,
        "n_features": len(features),
        "status": "ready_for_staging"
    }
    
    print("\n✅ Modelo registrado en MLflow Registry")
    print("\n📊 Detalles del modelo:")
    for key, value in model_info.items():
        print(f"  {key}: {value}")
    
    print("\n🚀 Próximos pasos:")
    print("  1. Deploy a Staging endpoint")
    print("  2. Run integration tests")
    print("  3. A/B test (90% old, 10% new)")
    print("  4. Promote to Production if successful")
else:
    print("\n⚠️ Registro cancelado")
    print("  El modelo no pasó validación")
    print("  Primero debe mejorar las métricas")

---

## 📊 Paso 7: Configurar Monitoreo

**Objetivo**: Setup de monitoreo en producción

**Componentes**:
* Data drift detection (PSI)
* Model performance tracking
* Alertas automáticas
* Dashboard de métricas

**Frecuencia**: Daily batch monitoring

In [0]:
print("📊 Paso 7: Configurar Monitoreo\n" + "="*60)

monitoring_config = {
    "drift_detection": {
        "method": "PSI",
        "threshold": 0.2,
        "features": features
    },
    "performance_tracking": {
        "metrics": ["accuracy", "f1_score", "precision", "recall"],
        "baseline_accuracy": accuracy,
        "baseline_f1": f1,
        "alert_threshold": 0.05  # 5% degradation
    },
    "schedule": {
        "frequency": "daily",
        "time": "02:00 UTC"
    },
    "alerts": {
        "channels": ["email", "slack"],
        "recipients": ["ml-team@company.com"]
    }
}

print("\n✅ Configuración de monitoreo:")
print("\n🔍 Drift Detection:")
print(f"  Método: {monitoring_config['drift_detection']['method']}")
print(f"  Threshold: {monitoring_config['drift_detection']['threshold']}")
print(f"  Features monitoreadas: {len(monitoring_config['drift_detection']['features'])}")

print("\n📊 Performance Tracking:")
for metric in monitoring_config['performance_tracking']['metrics']:
    print(f"  • {metric}")

print("\n🔔 Alertas:")
for channel in monitoring_config['alerts']['channels']:
    print(f"  • {channel}")

print("\n⏰ Schedule:")
print(f"  Frecuencia: {monitoring_config['schedule']['frequency']}")
print(f"  Hora: {monitoring_config['schedule']['time']}")

print("\n✅ Monitoreo configurado exitosamente")

---

## ✅ Resumen del Pipeline End-to-End

### 🎯 Pipeline Completo Implementado

| Paso | Layer/Fase | Output | Estado |
|------|------------|--------|--------|
| 1 | 🟫 Bronze | Datos crudos ingestados | ✅ |
| 2 | 🥈 Silver | Datos limpios y validados | ✅ |
| 3 | 🥇 Gold | Features de negocio | ✅ |
| 4 | 🏋️‍♂️ Training | Modelo con MLflow tracking | ✅ |
| 5 | ✅ Validation | Criterios de calidad | ✅ |
| 6 | 📦 Registry | Modelo versionado | ✅ |
| 7 | 📊 Monitoring | Setup de alertas | ✅ |

### 💡 Conceptos Aprendidos

1. **Medallion Architecture** (Bronze-Silver-Gold)
   * Separación de responsabilidades
   * Incrementalidad y reproducibilidad
   * Calidad de datos progresiva

2. **MLflow Experiment Tracking**
   * Logging de parámetros y métricas
   * Trazabilidad completa
   * Facilita comparación de experimentos

3. **Model Validation Gates**
   * Criterios de calidad antes de deployment
   * Previene modelos malos en producción
   * Automatización de decisión

4. **Production Monitoring**
   * Drift detection
   * Performance tracking
   * Alertas proactivas

### 🚀 Próximos Pasos en Producción

* Deploy a Staging endpoint
* Integration tests
* A/B testing (shadow mode)
* Gradual rollout a Producción
* Setup de reentrenamiento automático

---

**Universidad del Aconcagua 🇦🇷**